In [16]:
"""
Generalized code for computing the transition matrix on GL(n, F_q).

This computes a matrix related to random walks on flag varieties,
using Bruhat decomposition and unipotent subgroups.
"""

import itertools
from sage.all import *

# Parameters - change these as needed
q = 2
n = 3

# Setup
G = GL(n, GF(q))
R = LaurentPolynomialRing(QQ, 'v')
v = R.gen()

# ============================================================
# Basic predicates for matrix classification
# ============================================================

def is_upper_triangular(a):
    """Check if matrix is upper triangular."""
    m = matrix(a)
    return all(m[i, j] == 0 for i in range(n) for j in range(i))

def is_unipotent_upper_triangular(a):
    """Check if matrix is unipotent upper triangular (1s on diagonal, 0s below)."""
    m = matrix(a)
    upper_triangular = all(m[i, j] == 0 for i in range(n) for j in range(i))
    diagonal_ones = all(m[i, i] == 1 for i in range(n))
    return upper_triangular and diagonal_ones

def is_permutation_matrix(a):
    """Check if matrix is a permutation matrix."""
    m = matrix(a)
    for i in range(n):
        row_entries = [m[i, j] for j in range(n)]
        if row_entries.count(1) != 1 or row_entries.count(0) != n - 1:
            return False
    for j in range(n):
        col_entries = [m[i, j] for i in range(n)]
        if col_entries.count(1) != 1 or col_entries.count(0) != n - 1:
            return False
    return True

# ============================================================
# Build subgroups: B (Borel), U (unipotent), W (Weyl group)
# ============================================================

print(f"Building subgroups for GL({n}, F_{q})...")
print(f"|GL({n}, F_{q})| = {G.order()}")

B = [a for a in G if is_upper_triangular(matrix(a))]
U = [a for a in G if is_unipotent_upper_triangular(matrix(a))]
W = [a for a in G if is_permutation_matrix(matrix(a))]

print(f"|B| = {len(B)}")
print(f"|U| = {len(U)}")
print(f"|W| = {len(W)} (should be {factorial(n)})")

# ============================================================
# Order W by Bruhat order (length function)
# ============================================================

def permutation_from_matrix(m):
    """Extract permutation (as a list) from permutation matrix."""
    perm = []
    mat = matrix(m)
    for i in range(n):
        for j in range(n):
            if mat[i, j] == 1:
                perm.append(j + 1)  # 1-indexed for Sage Permutation
                break
    return Permutation(perm)

def length_of_permutation_matrix(w):
    """Compute the length (number of inversions) of a permutation matrix."""
    perm = permutation_from_matrix(w)
    return perm.inversions().__len__()

# Sort W by length (Bruhat order compatible)
ordW = sorted(W, key=lambda w: (length_of_permutation_matrix(w), 
                                 tuple(permutation_from_matrix(w))))

print(f"\nWeyl group elements ordered by length:")
for i, w in enumerate(ordW):
    perm = permutation_from_matrix(w)
    print(f"  w_{i}: {perm} (length {length_of_permutation_matrix(w)})")

# ============================================================
# Build orbit dictionary (G/B = flag variety)
# ============================================================

print(f"\nBuilding coset representatives for G/B...")

orbit_dict = {}
for g in G:
    # Check if g*b is already a representative for some b in B
    found_rep = None
    for rep in orbit_dict:
        # g and rep are in same coset iff g^{-1}*rep in B
        if is_upper_triangular(matrix(g.inverse() * rep)):
            found_rep = rep
            break
    
    if found_rep is not None:
        orbit_dict[found_rep].append(g)
    else:
        orbit_dict[g] = [g]

print(f"Number of flags (cosets G/B): {len(orbit_dict)}")
print(f"Expected: [n]_q! = {prod([(q^i - 1) for i in range(1, n+1)]) // prod([(q-1) for _ in range(n)])}")

# ============================================================
# Build B\G/B double coset dictionary (Bruhat cells)
# ============================================================

print(f"\nBuilding Bruhat cells (B\\G/B)...")

biorbit_dict = {tuple(map(tuple, matrix(w))): [] for w in ordW}

for w in ordW:
    w_key = tuple(map(tuple, matrix(w)))
    for rep in orbit_dict:
        # Check if rep is in BwB
        # rep in BwB iff exists b1, b2 in B with rep = b1 * w * b2
        # Equivalently, check Bruhat decomposition
        for u in U:
            if is_upper_triangular(matrix((u * w).inverse() * rep)):
                if rep not in biorbit_dict[w_key]:
                    biorbit_dict[w_key].append(rep)
                break

for w in ordW:
    w_key = tuple(map(tuple, matrix(w)))
    perm = permutation_from_matrix(w)
    print(f"  |BwB/B| for w={perm}: {len(biorbit_dict[w_key])}")

# ============================================================
# Compute U_w = U ∩ wUw^{-1} for each w
# ============================================================

def compute_Uw(w):
    """Compute U_w = {u in U : w^{-1} u w in U}."""
    w_inv = w.inverse()
    return [u for u in U if is_unipotent_upper_triangular(matrix(w_inv * u * w))]

Uw_dict = {tuple(map(tuple, matrix(w))): compute_Uw(w) for w in ordW}

print(f"\nSizes of U_w:")
for w in ordW:
    w_key = tuple(map(tuple, matrix(w)))
    perm = permutation_from_matrix(w)
    print(f"  |U_w| for w={perm}: {len(Uw_dict[w_key])}")

# ============================================================
# Compute weights for elements of U
# This generalizes the hardcoded weights
# ============================================================

def compute_weight(u):
    """
    Compute weight of unipotent element u.
    Weight = number of nonzero entries above diagonal.
    This corresponds to the 'height' in the unipotent radical.
    """
    m = matrix(u)
    count = 0
    for i in range(n):
        for j in range(i + 1, n):
            if m[i, j] != 0:
                count += 1
    return count

weights = {tuple(map(tuple, matrix(u))): compute_weight(u) for u in U}

print(f"\nWeight distribution in U:")
weight_counts = {}
for u in U:
    w = weights[tuple(map(tuple, matrix(u)))]
    weight_counts[w] = weight_counts.get(w, 0) + 1
for w in sorted(weight_counts.keys()):
    print(f"  Weight {w}: {weight_counts[w]} elements")

# ============================================================
# Fixed flags computation
# ============================================================

def fixed_flags(a):
    """
    Return list of flag representatives fixed by a.
    A flag gB is fixed by a iff g^{-1} a g in B.
    """
    return [g for g in orbit_dict if is_upper_triangular(matrix(g.inverse() * a * g))]

def cap_with_w(flag_set, w):
    """
    Return flags in flag_set that lie in the Bruhat cell BwB/B.
    """
    w_key = tuple(map(tuple, matrix(w)))
    return [x for x in flag_set if x in biorbit_dict[w_key]]

# ============================================================
# Signature function
# ============================================================

def signature(a):
    """
    Compute the signature of element a.
    Returns a list indexed by ordW, with entries v^k where k = log_q(count).
    """
    fix = fixed_flags(a)
    sig = []
    for w in ordW:
        count = len(cap_with_w(fix, w))
        if count == 0:
            sig.append(R(0))
        else:
            # count should be a power of q
            if count == 1:
                k = 0
            else:
                k = ZZ(count).log(q)
            sig.append(v^k)
    return sig

# ============================================================
# Normalization helper
# ============================================================

def normalize(lst):
    """Normalize a list so entries sum to 1."""
    s = sum(lst)
    if s == 0:
        return lst
    return [x / s for x in lst]

# Build the transition matrices
print(f"\nBuilding transition matrices...")

# Create fraction field for the computations
F = R.fraction_field()

# Matrix m1: W x U matrix
# Entry (w, u) is (v-1)^weight(u) if u in U_w, else 0, then normalize rows
m1 = []
for w in ordW:
    w_key = tuple(map(tuple, matrix(w)))
    row = []
    for u in U:
        u_key = tuple(map(tuple, matrix(u)))
        if u in Uw_dict[w_key]:
            row.append(F((v - 1) ** weights[u_key]))
        else:
            row.append(F(0))
    m1.append(normalize(row))

# Matrix m2: U x W matrix
# Each row is the normalized signature of that u
m2 = []
for u in U:
    sig = signature(u)
    # Convert to fraction field
    sig_F = [F(x) for x in sig]
    m2.append(normalize(sig_F))

# Final matrix: m = m1 * m2, a W x W transition matrix
m1_matrix = matrix(F, m1)
m2_matrix = matrix(F, m2)
m = m1_matrix * m2_matrix
print(m)
print()
mv2 = m.subs(v=2)
print(mv2)
print(mv2.eigenvalues())

Building subgroups for GL(3, F_2)...
|GL(3, F_2)| = 168
|B| = 8
|U| = 8
|W| = 6 (should be 6)

Weyl group elements ordered by length:
  w_0: [1, 2, 3] (length 0)
  w_1: [1, 3, 2] (length 1)
  w_2: [2, 1, 3] (length 1)
  w_3: [2, 3, 1] (length 2)
  w_4: [3, 1, 2] (length 2)
  w_5: [3, 2, 1] (length 3)

Building coset representatives for G/B...
Number of flags (cosets G/B): 21
Expected: [n]_q! = 21

Building Bruhat cells (B\G/B)...
  |BwB/B| for w=[1, 2, 3]: 1
  |BwB/B| for w=[1, 3, 2]: 2
  |BwB/B| for w=[2, 1, 3]: 2
  |BwB/B| for w=[2, 3, 1]: 4
  |BwB/B| for w=[3, 1, 2]: 4
  |BwB/B| for w=[3, 2, 1]: 8

Sizes of U_w:
  |U_w| for w=[1, 2, 3]: 8
  |U_w| for w=[1, 3, 2]: 4
  |U_w| for w=[2, 1, 3]: 4
  |U_w| for w=[2, 3, 1]: 2
  |U_w| for w=[3, 1, 2]: 2
  |U_w| for w=[3, 2, 1]: 1

Weight distribution in U:
  Weight 0: 1 elements
  Weight 1: 3 elements
  Weight 2: 3 elements
  Weight 3: 1 elements

Building transition matrices...
[                                          v^3/(v^3 + 2*v^2 + 2